<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W7D3Exercises_XP_RAG_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [ ]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

In [ ]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [ ]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={"source": row[source_column]}
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])

## 2) Split into chunks


In [ ]:
chunk_size = 512 # A common starting point for chunk size
chunk_overlap = 50 # A small percentage of chunk size

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

splits = splitter.split_documents(documents)
print("Chunks:", len(splits))
print("First chunk:", splits[0].metadata)
print(splits[0].page_content[:350])

## 3) Vector store + retriever (FAISS)


In [ ]:
from langchain_community.vectorstores import FAISS, DistanceStrategy

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

vectorstore = FAISS.from_documents(
    splits, embeddings, distance_strategy=DistanceStrategy.COSINE
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready")

## 4) Build the RAG chain


In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA  # on latest stack

llm_id = "google/flan-t5-small"
hf = pipeline(
    "text-generation", # Changed from "text2text-generation" to "text-generation"
    model=llm_id,
    max_new_tokens=50 # limiting token generation to prevent long outputs
)

llm = HuggingFacePipeline(pipeline=hf)

qa = RetrievalQA.from_chain_type(
    llm=llm, retriever=retriever, chain_type="stuff"
)

print("RAG chain ready")

## 5) Demo: RAG vs no-RAG


In [ ]:
q = "How can I retrieve a model from the Hugging Face Hub?"

# ── No-RAG : LLM seul, sans contexte externe ─────────────────────────────────
no_rag_prompt = (
    "Answer the question. If you are not sure, say you are not sure.\n\n"
    f"Question: {q}\n"
    "Answer:"
)
no_rag_answer = hf(no_rag_prompt)[0]["generated_text"]

# ── RAG : LLM + contexte récupéré depuis FAISS ───────────────────────────────
rag_result = qa({"query": q})  # On passe la question via la clé "query"

# ── Affichage comparatif ──────────────────────────────────────────────────────
print("=" * 60)
print(f"Q: {q}")
print("=" * 60)
print("\n🔴 No-RAG answer (LLM seul) :")
print(" ", no_rag_answer)

print("\n🟢 RAG answer (LLM + contexte récupéré) :")
print(" ", rag_result["result"])

print("\n📚 Sources utilisées par le RAG :")
for d in rag_result["source_documents"]:
    print("  -", d.metadata.get("source", "N/A"))